In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/anonymous111111111/responses-sheet/Doha Evaluation Form (Responses) (1).xlsx
/kaggle/input/datasets/kgan31/complete-kavita-dataset/kavitas_merged.csv


In [8]:
import pandas as pd
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Ensure tokenizer is ready
nltk.download('punkt', quiet=True)

def compute_novelty(target_dohas, reference_corpus):
    """
    Computes 1 - Average BLEU score.
    If target_dohas and reference_corpus are the same object (Self-BLEU), 
    it excludes the current item from references.
    """
    smoothie = SmoothingFunction().method1
    total_score = 0
    n = len(target_dohas)
    
    # Tokenize strings into word lists
    target_tok = [str(d).split() for d in target_dohas]
    ref_tok = [str(d).split() for d in reference_corpus]

    # Check if we are doing Self-BLEU (comparing list to itself)
    is_self_comparison = (target_dohas is reference_corpus)

    for i in range(n):
        hypothesis = target_tok[i]
        
        if is_self_comparison:
            # For Self-BLEU, don't compare a doha against itself
            references = [ref_tok[j] for j in range(n) if i != j]
        else:
            # Compare generated doha against the entire training corpus
            references = ref_tok
            
        score = sentence_bleu(references, hypothesis, smoothing_function=smoothie)
        total_score += score

    avg_bleu = total_score / n
    return 1 - avg_bleu

if __name__ == '__main__':
    # 1. Your list of 50 dohas (Paste your list here)
    generated_dohas = [
        "कोयल की वाणी करे, मन में रस समान। सुरभित करती है सदा , मीठे-मन की मुस्कान ॥",
        "मिट्टी का पानी नहीं, तरु-से शीतल नीर। धरती पर जाती छाती पड़ीं, माटी की धार ॥",
        "जब लगि बिधि गंगाजली, मन की हौं न मन खाय। समझ हरि के पीर ते, ‘रज’ आपु में लाभ ॥",
        "मोर उड़त मन खोलिन में ललित बालन ओर। मनो बरुन बाल के, छवि लाल स्याम ॥",
        "नैनन दृग मुसुमा बदन, लालकानि लसत मुसकान। परी छवि बाल तिय मुख, लखियत दृग नैन ॥",
        "पिय के हेत अति अधिक, अरी तनत नहिं ठौर। छरि पियत आइ कै, अब कछु आनु दरकार ॥",
        "कबीर नीरस की नदी, बिन मन को धार। बिना नेह पिया न हो कभी, प्यास नहीं आस ॥",
        "अब तो मरमला के गयो, अब न काऊ की कोय। अब भी आवत ले गए, अब तो नहिं रोय ॥",
        "धरनी गुन गुप्त है, रहनि एक संसार। परमातम जग में गनत, आचार रूप ॥",
        "जो निज को आपनो, दीजे हाथ चलाय। तासों पल में ही नहीं, वहै न कोई कोय ॥",
        "गुरु गुरु से ज्ञान नहि, जाके बझो होय। धरनी शब्द कहैं मिले, गुरुको आपु ज्ञान ॥",
        "फलफल फल फल करि करै, धरनी जो फल पाय। कर्म करम परहिँ नहि कीजिये, आपहिं आपु देय ॥",
        "मिलन बसते मिलत है प्रिय के मिलन की बात। मिलने में पिय आ बसे हिय आवत आनन्द ॥",
        "अँधियारे में बसे, सूरज है दीप। उजड़ी रात गई, तम का दीपक जाग ॥",
        "ऐसा था था नहीं, जीवन का रूप। कोई भी एक है कहाँ, होता यह अज्ञान ॥",
        "तुम मित्रता में बसा, मन बसिए बसन्त। ऋतु की घन धूप की खिले, फूल खिलें फूल ॥",
        "कबीर माया को तजि, छोड़ौं उर-छाह। जो मन में लाछे भलो, भक्ति कर मन-रूप ॥",
        "धरनी निगरज जीव को, सो सुख दुख न कोय। सहजै जन दुख सुख दुख दुख बां, वासो साधु भाय ॥",
        "हाथ बहिँ अपने साथ नित, धरनी जोह लेहु। तजि मानै आपु लेत, अपनीहिँ देहिं ॥",
        "निन्दा निज पग चलत, छूटन मन के ठौर। नाचे मन में बसो न रुक्यो, जानों हित प्रान ॥",
        "धरनी जब लगि बिरह है, निष्ठुर अधीर। तासे मरै पार लौं तबहि, तौ तरस नीर ॥",
        "फिर से आज भी देख लो, मन्द-फिर भीम। कुछ अधिक शक्ति को छके, अति बढ़ा रहे हैं आज ॥",
        "नदी नदी के पार, फिर तोरा करती प्रीत। नाव नदिया उस अड़ी, कभी न तोड़ें पतवार ॥",
        "पारस नदी का अजल, हेती सागर पार। पवन-धर सी सी हुई, बहती धार ॥",
        "कोई पता है मुझे, कुछ नहीं किसी का नाम। अपना ही है यहाँ, कहीं न जाने नाम ॥",
        "रहिमन एक पत्थर कहुँ, आपु बस लेहु बुझाय। जो हृदइ के पग ते भले, जाहिं उठि जाय ॥",
        "नंग-सोये कोटि दो, सबु बनाइ के साथ। इन श्मशान में चलत है, काहू कोटि देत ॥",
        "बाल ऊड़ि नीच परै, मन पै इतराय। चढ़त-परमन मारिये, समुझि चढ़्यौ जाय ॥",
        "दिल में एक ही नहीं, कोई कौन किसलय का ठौर। वह रहा है, कहाँ कहीं एक ॥",
        "सागर के जल में उठे, काले आकाश। जैसे पर्वत पार हैं, मोती सागर का छोर ॥",
        "दीपक सूरज का करे, दीपों की किरण। साँझ सुबह से हो रहे, तम-दिन अँधियारे रात ॥",
        "नदी पहाड़ चढ़ता नहीं, पर्वत नदिया धारा। पत्थर मीन सूखती, रेत गये सब रेत ॥",
        "जो जानै सब जगत सो, होय न जानि सकैं कोय। अपने कोहू मानिया, तेहि कुंभ लेय ॥",
        "माटी की धूल में, भूखी रख एक फूल। फूल सभी फूलों से मिले, सी होता आकाश ॥",
        "दो राह के पार पर, जाना कठिनगार। छोड़ नहीं कोई नहीं, अपना सरवार ॥",
        "संसार में मत भूलिये, जग के जगत बसाय। इसमें सेवक बने, सब जीवन होय ॥",
        "देखत न घन की नज़र, जो देखौ तेरी खोज। नहिं दीखे ओर के, कोई कोई दोष ॥",
        "प्रेम प्रेम भक्ति नित, हृदय बसत हरष। परमातम पिय नेह बिन, मिले न मिलता प्यार ॥",
        "जब तोहि जलाइये, तब तकइ आपहिं। तुम ही दृगन सों भरी, इन आँखों तें जाहि ॥",
        "धरती पर तन भूमि है, अब भू की देह। काँटों के पंखुने से नहीं, पंख पंखों की घास ॥",
        "मूल्य व्याकरण से हुआ, वहाँ सा मूल यह रूप। जहाँ कहीं से होता रहा, वह ही एक अंग ॥",
        "दिन भर की नींद से, थी दिन भर भी जाग। रवि-दिन तुम रख कर सके, सूर्य से अपने हाथ ॥",
        "नदी सागर-हर धार का, पार पार धार। जल जल में भर रही, मर गया पार ॥",
        "पत्थर बिन केवल नहीं, बिना बियोग संसार। धरनी तजै आपनाइया, ईश्वर का वास ॥",
        "पवन चले चलत जो सदा, जहाँ और जगह न कोय। गति हवा नदी को देख कर, पवन रही देय ॥",
        "धरती है बूँद-सा, जल है सागर समान। धरती पर जलधार है, धरती शीतल बरसात ॥",
        "धरती से देह सी, काया कातरित अंग। कबीर कुल्हाड़ी में पड़े, जल में पानी चंग ॥",
        "दोपहर हवा छाँव है, चाँदनी सु शीतल धूप। छाया दिन भर दे रही, दो पल शीतल दीप ॥",
        "दीपो दीपक जलता सदा, जले रात दिन रात। अँधियारे से सूर्य से, जगमगा उजियार ॥",
        "पोटली नयन की पटी, मन आइल सब मीत। बसत तेरे रंग है, होन मित्र सुजान ॥",
        "नदी पारै पारस की, गहि नाव तीर। पिय-मपधारि गई अधीर को, चली सैन ॥",
        "अब तक है एक लौं जल, आगत ही जल धार। कमल कीरी कमल में, धरि परै जल धार ॥",
        "कपट कथरा चलत बकरी, कर्म कोटि काठ। कहैं कबीर सो सफेद है, फिर न को"
    ]

    # 2. Path to your reference dataset (the original 7,800 dohas)
    TRAIN_DATA_PATH = '/kaggle/input/datasets/anonymous111111111/doha-dataset/dohas_final_hindi_dataset.csv'

    try:
        # Load the large dataset from CSV
        df_train = pd.read_csv(TRAIN_DATA_PATH)
        train_corpus = df_train['Doha'].dropna().tolist()

        print(f"Loaded {len(train_corpus)} training dohas.")
        print(f"Analyzing {len(generated_dohas)} generated dohas...")

        # 3. Compute Scores
        # External Novelty: How new are these compared to the training data?
        ns_ext = compute_novelty(generated_dohas, train_corpus)
        
        # Self-Novelty: How diverse are these 50 dohas among themselves?
        ns_self = compute_novelty(generated_dohas, generated_dohas)

        # 4. Final Output
        print("\n" + "="*50)
        print("METRICS FOR INLP REPORT")
        print("="*50)
        print(f"External Novelty Score (NS_ext): {ns_ext:.4f}")
        print(f"Internal Novelty Score (NS_self): {ns_self:.4f}")
        print("-" * 50)
        print("Metric Guide:")
        print(" - NS_ext close to 1.0: Model is highly creative (not copying).")
        print(" - NS_self close to 1.0: Model outputs are very diverse.")
        print("="*50)

    except FileNotFoundError:
        print(f"Error: Could not find the dataset at {TRAIN_DATA_PATH}")
    except Exception as e:
        print(f"An error occurred: {e}")

Loaded 8191 training dohas.
Analyzing 53 generated dohas...

METRICS FOR INLP REPORT
External Novelty Score (NS_ext): 0.9257
Internal Novelty Score (NS_self): 0.9729
--------------------------------------------------
Metric Guide:
 - NS_ext close to 1.0: Model is highly creative (not copying).
 - NS_self close to 1.0: Model outputs are very diverse.


In [7]:
import pandas as pd
import unicodedata
from tqdm import tqdm

# Unicode Constants
HALANT       = '\u094D'
ANUSVARA     = '\u0902'
CHANDRABINDU = '\u0901'
VISARGA      = '\u0903'
NUKTA        = '\u093C'

SWAR_WEIGHT = {
    '\u0905': 1, '\u0906': 2, '\u0907': 1, '\u0908': 2, '\u0909': 1, '\u090A': 2,
    '\u090B': 1, '\u090C': 1, '\u090F': 2, '\u0910': 2, '\u0913': 2, '\u0914': 2,
}
MATRA_WEIGHT = {
    '\u093E': 2, '\u093F': 1, '\u0940': 2, '\u0941': 1, '\u0942': 2, '\u0943': 1,
    '\u0947': 2, '\u0948': 2, '\u094B': 2, '\u094C': 2,
}

def is_consonant(ch):
    cp = ord(ch)
    return (0x0915 <= cp <= 0x0939) or (0x0958 <= cp <= 0x095F)

def tokenize(word):
    word = unicodedata.normalize('NFC', word)
    tokens = []
    chars = list(word)
    i = 0
    n = len(chars)

    while i < n:
        ch = chars[i]

        # 1. Handle Vowels
        if ch in SWAR_WEIGHT:
            weight = SWAR_WEIGHT[ch]
            unit = ch
            i += 1
            while i < n and chars[i] in (ANUSVARA, VISARGA):
                weight = 2 # Anusvara/Visarga makes it Guru
                unit += chars[i]
                i += 1
            tokens.append({'unit': unit, 'weight': weight})

        # 2. Handle Consonants
        elif is_consonant(ch):
            unit = ch
            i += 1
            if i < n and chars[i] == NUKTA:
                unit += chars[i]; i += 1

            # CONJUNCT RULE: Halant makes the PREVIOUS syllable Guru
            if i < n and chars[i] == HALANT:
                unit += chars[i]; i += 1
                if tokens:
                    tokens[-1]['weight'] = 2
                tokens.append({'unit': unit, 'weight': 0})
            else:
                # Check for Matras
                matra_w = 1 # Default inherent 'a'
                if i < n and chars[i] in MATRA_WEIGHT:
                    matra_w = MATRA_WEIGHT[chars[i]]
                    unit += chars[i]; i += 1

                # Check for Anusvara/Visarga on consonant
                while i < n and chars[i] in (ANUSVARA, VISARGA, CHANDRABINDU):
                    if chars[i] in (ANUSVARA, VISARGA): matra_w = 2
                    unit += chars[i]; i += 1
                tokens.append({'unit': unit, 'weight': matra_w})
        else:
            i += 1 # Ignore non-devanagari
    return tokens

def count_matra(text):
    return sum(t['weight'] for t in tokenize(text))

def parse_single_line(line, has_comma=None):
    """
    Parse a single line and return its charan matras.
    
    Args:
        line: A single line of text
        has_comma: If None, auto-detect; if True/False, use that mode
    
    Returns:
        List of matra counts for charans in this line
    """
    line = line.strip()
    if not line:
        return []
    
    # Auto-detect comma in this line if not specified
    if has_comma is None:
        has_comma = ',' in line
    
    charan_matras = []
    
    if has_comma:
        # Parse WITH comma delimiters
        parts = [p.strip() for p in line.split(',') if p.strip()]
        for part in parts:
            charan_matras.append(count_matra(part))
    else:
        # Parse WITHOUT commas: word-by-word accumulation
        words = line.split()  # Split by whitespace
        current_charan_matra = 0
        
        for word in words:
            word_matra = count_matra(word)
            current_charan_matra += word_matra
            
            # If we've reached or exceeded 13 matras, finalize this charan
            if current_charan_matra >= 13:
                charan_matras.append(current_charan_matra)
                current_charan_matra = 0
        
        # Append any remaining matras as a charan (if not zero)
        if current_charan_matra > 0:
            charan_matras.append(current_charan_matra)
    
    return charan_matras

def get_charan_matras(doha_text):
    """
    Parse doha LINE-BY-LINE, checking each line independently for comma presence.
    
    Args:
        doha_text: The full doha string
    
    Returns:
        List of matra counts for all charans across all lines
    """
    text = doha_text.replace('॥', '।').strip()
    lines = [l.strip() for l in text.split('।') if l.strip()]
    
    charan_matras = []
    
    for line in lines:
        # Each line is checked independently for comma presence
        line_matras = parse_single_line(line, has_comma=None)  # None = auto-detect per line
        charan_matras.extend(line_matras)
    
    return charan_matras

def has_comma_in_line(line):
    """
    Check if a single line has comma delimiter.
    
    Returns:
        True if comma found in this line, False otherwise
    """
    return ',' in line.strip()

def compute_mas(cm):
    """Compute Mean Absolute Squared error from ideal charan structure"""
    ideal = [13, 11, 13, 11]
    # Padding with 0 if charans are missing
    cm4 = (cm + [0]*4)[:4]
    return sum(abs(cm4[i] - ideal[i]) for i in range(4))


# ============================================================================
# MAIN: Process External CSV File
# ============================================================================

if __name__ == '__main__':
    # 1. SET YOUR PATHS
    # Update this path to your actual CSV file location
    INPUT_CSV = '/kaggle/input/datasets/anonymous111111111/doha-dataset/dohas_final_hindi_dataset.csv'  # Change this to your file path
    OUTPUT_CSV = 'doha_matra_analysis_results.csv'

    try:
        # 2. Load the Dataset
        print(f"Loading dataset from: {INPUT_CSV}")
        df = pd.read_csv(INPUT_CSV)
        
        # Check what columns are available
        print(f"\nAvailable columns: {df.columns.tolist()}")
        
        # Find the column that contains dohas (common names: 'Doha', 'doha', 'text', 'content')
        doha_column = None
        for col in ['Doha', 'doha', 'text', 'content', 'Text', 'Content']:
            if col in df.columns:
                doha_column = col
                break
        
        if doha_column is None:
            print(f"Error: Could not find doha column. Available columns: {df.columns.tolist()}")
            exit(1)
        
        print(f"Using column '{doha_column}' for doha text\n")
        
        # Ensure we only process rows where doha column is not empty
        df = df.dropna(subset=[doha_column])
        
        all_results = []
        print(f"Processing {len(df)} dohas...\n")

        # 3. Iterate through the DataFrame
        for index, row in tqdm(df.iterrows(), total=len(df), desc="Processing Dohas"):
            try:
                doha_text = str(row[doha_column])
                
                # Use your existing parsing logic
                cm = get_charan_matras(doha_text)
                mas = compute_mas(cm)
                
                # Ensure we have 4 values for storage
                cm4 = (cm + [0]*4)[:4]
                total_matras = sum(cm4)
                
                # Store results in a dictionary
                all_results.append({
                    'm1': cm4[0],
                    'm2': cm4[1],
                    'm3': cm4[2],
                    'm4': cm4[3],
                    'Total_Matras': total_matras,
                    'MAS': mas,
                    'Is_Perfect': 1 if mas == 0 else 0
                })
            except Exception as e:
                # Handle errors in individual dohas gracefully
                print(f"\nWarning: Error processing row {index}: {e}")
                all_results.append({
                    'm1': 0,
                    'm2': 0,
                    'm3': 0,
                    'm4': 0,
                    'Total_Matras': 0,
                    'MAS': -1,  # Use -1 to indicate error
                    'Is_Perfect': 0
                })

        # 4. Merge results back into the DataFrame
        results_df = pd.DataFrame(all_results)
        final_df = pd.concat([df.reset_index(drop=True), results_df], axis=1)

        # 5. Calculate Final Global Stats (excluding errors with MAS = -1)
        valid_results = final_df[final_df['MAS'] >= 0]
        
        avg_mas = valid_results['MAS'].mean()
        avg_matras = valid_results['Total_Matras'].mean()
        perfect_count = valid_results['Is_Perfect'].sum()
        error_count = len(final_df) - len(valid_results)

        print("\n" + "="*60)
        print("GLOBAL DATASET SUMMARY")
        print("="*60)
        print(f"Total Dohas Analyzed     : {len(final_df)}")
        print(f"Successfully Processed   : {len(valid_results)}")
        print(f"Errors Encountered       : {error_count}")
        print(f"Average Matra Count      : {avg_matras:.2f}")
        print(f"Average MAS              : {avg_mas:.2f}")
        print(f"Perfect Dohas (0 MAS)    : {perfect_count} ({(perfect_count/len(valid_results))*100:.2f}%)")
        print("="*60)
        
        # Additional statistics
        print("\nMATRA DISTRIBUTION:")
        print(f"  M1 (avg): {valid_results['m1'].mean():.2f} (ideal: 13)")
        print(f"  M2 (avg): {valid_results['m2'].mean():.2f} (ideal: 11)")
        print(f"  M3 (avg): {valid_results['m3'].mean():.2f} (ideal: 13)")
        print(f"  M4 (avg): {valid_results['m4'].mean():.2f} (ideal: 11)")
        
        print("\nMAS DISTRIBUTION:")
        mas_stats = valid_results['MAS'].describe()
        print(f"  Min MAS: {mas_stats['min']:.0f}")
        print(f"  Max MAS: {mas_stats['max']:.0f}")
        print(f"  Std Dev: {mas_stats['std']:.2f}")
        print("="*60)

        # 6. Save to CSV
        final_df.to_csv(OUTPUT_CSV, index=False)
        print(f"\n✓ Detailed results saved to: {OUTPUT_CSV}")
        print(f"✓ Output file has {len(final_df)} rows and {len(final_df.columns)} columns")

    except FileNotFoundError:
        print(f"\n✗ Error: Could not find file at '{INPUT_CSV}'")
        print(f"  Please check the file path and try again.")
    except Exception as e:
        print(f"\n✗ An error occurred: {e}")
        import traceback
        traceback.print_exc()

Loading dataset from: /kaggle/input/datasets/anonymous111111111/doha-dataset/dohas_final_hindi_dataset.csv

Available columns: ['Author', 'Doha', 'Theme', 'Context', 'URL']
Using column 'Doha' for doha text

Processing 8191 dohas...



Processing Dohas: 100%|██████████| 8191/8191 [00:00<00:00, 12620.09it/s]



GLOBAL DATASET SUMMARY
Total Dohas Analyzed     : 8191
Successfully Processed   : 8191
Errors Encountered       : 0
Average Matra Count      : 47.80
Average MAS              : 4.55
Perfect Dohas (0 MAS)    : 3644 (44.49%)

MATRA DISTRIBUTION:
  M1 (avg): 13.00 (ideal: 13)
  M2 (avg): 12.13 (ideal: 11)
  M3 (avg): 12.70 (ideal: 13)
  M4 (avg): 9.97 (ideal: 11)

MAS DISTRIBUTION:
  Min MAS: 0
  Max MAS: 97
  Std Dev: 9.04

✓ Detailed results saved to: doha_matra_analysis_results.csv
✓ Output file has 8191 rows and 12 columns


In [8]:
import unicodedata

# Unicode Constants
HALANT       = '\u094D'
ANUSVARA     = '\u0902'
CHANDRABINDU = '\u0901'
VISARGA      = '\u0903'
NUKTA        = '\u093C'

SWAR_WEIGHT = {
    '\u0905': 1, '\u0906': 2, '\u0907': 1, '\u0908': 2, '\u0909': 1, '\u090A': 2,
    '\u090B': 1, '\u090C': 1, '\u090F': 2, '\u0910': 2, '\u0913': 2, '\u0914': 2,
}
MATRA_WEIGHT = {
    '\u093E': 2, '\u093F': 1, '\u0940': 2, '\u0941': 1, '\u0942': 2, '\u0943': 1,
    '\u0947': 2, '\u0948': 2, '\u094B': 2, '\u094C': 2,
}

def is_consonant(ch):
    cp = ord(ch)
    return (0x0915 <= cp <= 0x0939) or (0x0958 <= cp <= 0x095F)

def tokenize(word):
    word = unicodedata.normalize('NFC', word)
    tokens = []
    chars = list(word)
    i = 0
    n = len(chars)

    while i < n:
        ch = chars[i]

        # 1. Handle Vowels
        if ch in SWAR_WEIGHT:
            weight = SWAR_WEIGHT[ch]
            unit = ch
            i += 1
            while i < n and chars[i] in (ANUSVARA, VISARGA):
                weight = 2 # Anusvara/Visarga makes it Guru
                unit += chars[i]
                i += 1
            tokens.append({'unit': unit, 'weight': weight})

        # 2. Handle Consonants
        elif is_consonant(ch):
            unit = ch
            i += 1
            if i < n and chars[i] == NUKTA:
                unit += chars[i]; i += 1

            # CONJUNCT RULE: Halant makes the PREVIOUS syllable Guru
            if i < n and chars[i] == HALANT:
                unit += chars[i]; i += 1
                if tokens:
                    tokens[-1]['weight'] = 2
                tokens.append({'unit': unit, 'weight': 0})
            else:
                # Check for Matras
                matra_w = 1 # Default inherent 'a'
                if i < n and chars[i] in MATRA_WEIGHT:
                    matra_w = MATRA_WEIGHT[chars[i]]
                    unit += chars[i]; i += 1

                # Check for Anusvara/Visarga on consonant
                while i < n and chars[i] in (ANUSVARA, VISARGA, CHANDRABINDU):
                    if chars[i] in (ANUSVARA, VISARGA): matra_w = 2
                    unit += chars[i]; i += 1
                tokens.append({'unit': unit, 'weight': matra_w})
        else:
            i += 1 # Ignore non-devanagari
    return tokens

def count_matra(text):
    return sum(t['weight'] for t in tokenize(text))

def parse_single_line(line, has_comma=None):
    """
    Parse a single line and return its charan matras.
    
    Args:
        line: A single line of text
        has_comma: If None, auto-detect; if True/False, use that mode
    
    Returns:
        List of matra counts for charans in this line
    """
    line = line.strip()
    if not line:
        return []
    
    # Auto-detect comma in this line if not specified
    if has_comma is None:
        has_comma = ',' in line
    
    charan_matras = []
    
    if has_comma:
        # Parse WITH comma delimiters
        parts = [p.strip() for p in line.split(',') if p.strip()]
        for part in parts:
            charan_matras.append(count_matra(part))
    else:
        # Parse WITHOUT commas: word-by-word accumulation
        words = line.split()  # Split by whitespace
        current_charan_matra = 0
        
        for word in words:
            word_matra = count_matra(word)
            current_charan_matra += word_matra
            
            # If we've reached or exceeded 13 matras, finalize this charan
            if current_charan_matra >= 13:
                charan_matras.append(current_charan_matra)
                current_charan_matra = 0
        
        # Append any remaining matras as a charan (if not zero)
        if current_charan_matra > 0:
            charan_matras.append(current_charan_matra)
    
    return charan_matras

def get_charan_matras(doha_text):
    """
    Parse doha LINE-BY-LINE, checking each line independently for comma presence.
    
    Args:
        doha_text: The full doha string
    
    Returns:
        List of matra counts for all charans across all lines
    """
    text = doha_text.replace('॥', '।').strip()
    lines = [l.strip() for l in text.split('।') if l.strip()]
    
    charan_matras = []
    
    for line in lines:
        # Each line is checked independently for comma presence
        line_matras = parse_single_line(line, has_comma=None)  # None = auto-detect per line
        charan_matras.extend(line_matras)
    
    return charan_matras

def has_comma_in_line(line):
    """
    Check if a single line has comma delimiter.
    
    Returns:
        True if comma found in this line, False otherwise
    """
    return ',' in line.strip()

def compute_mas(cm):
    """Compute Mean Absolute Squared error from ideal charan structure"""
    ideal = [13, 11, 13, 11]
    # Padding with 0 if charans are missing
    cm4 = (cm + [0]*4)[:4]
    return sum(abs(cm4[i] - ideal[i]) for i in range(4))

# --- Execution ---
doha_list = [
     "कोयल की वाणी करे, मन में रस समान। सुरभित करती है सदा , मीठे-मन की मुस्कान ॥",
    "मिट्टी का पानी नहीं, तरु-से शीतल नीर। धरती पर जाती छाती पड़ीं, माटी की धार ॥",
    "जब लगि बिधि गंगाजली, मन की हौं न मन खाय। समझ हरि के पीर ते, 'रज' आपु में लाभ ॥",
    "मोर उड़त मन खोलिन में ललित बालन ओर। मनो बरुन बाल के, छवि लाल स्याम ॥",
    "नैनन दृग मुसुमा बदन, लालकानि लसत मुसकान। परी छवि बाल तिय मुख, लखियत दृग नैन ॥",
    "पिय के हेत अति अधिक, अरी तनत नहिं ठौर। छरि पियत आइ कै, अब कछु आनु दरकार ॥",
    "कबीर नीरस की नदी, बिन मन को धार। बिना नेह पिया न हो कभी, प्यास नहीं आस ॥",
    "अब तो मरमला के गयो, अब न काऊ की कोय। अब भी आवत ले गए, अब तो नहिं रोय ॥",
    "धरनी गुन गुप्त है, रहनि एक संसार। परमातम जग में गनत, आचार रूप ॥",
    "जो निज को आपनो, दीजे हाथ चलाय। तासों पल में ही नहीं, वहै न कोई कोय ॥",
    "गुरु गुरु से ज्ञान नहि, जाके बझो होय। धरनी शब्द कहैं मिले, गुरुको आपु ज्ञान ॥",
    "फलफल फल फल करि करै, धरनी जो फल पाय। कर्म करम परहिँ नहि कीजिये, आपहिं आपु देय ॥",
    "मिलन बसते मिलत है प्रिय के मिलन की बात। मिलने में पिय आ बसे हिय आवत आनन्द ॥",
    "अँधियारे में बसे, सूरज है दीप। उजड़ी रात गई, तम का दीपक जाग ॥",
    "ऐसा था था नहीं, जीवन का रूप। कोई भी एक है कहाँ, होता यह अज्ञान ॥",
    "तुम मित्रता में बसा, मन बसिए बसन्त। ऋतु की घन धूप की खिले, फूल खिलें फूल ॥",
    "कबीर माया को तजि, छोड़ौं उर-छाह। जो मन में लाछे भलो, भक्ति कर मन-रूप ॥",
    "धरनी निगरज जीव को, सो सुख दुख न कोय। सहजै जन दुख सुख दुख दुख बां, वासो साधु भाय ॥",
    "हाथ बहिँ अपने साथ नित, धरनी जोह लेहु। तजि मानै आपु लेत, अपनीहिँ देहिं ॥",
    "निन्दा निज पग चलत, छूटन मन के ठौर। नाचे मन में बसो न रुक्यो, जानों हित प्रान ॥",
    "धरनी जब लगि बिरह है, निष्ठुर अधीर। तासे मरै पार लौं तबहि, तौ तरस नीर ॥",
    "फिर से आज भी देख लो, मन्द-फिर भीम। कुछ अधिक शक्ति को छके, अति बढ़ा रहे हैं आज ॥",
    "नदी नदी के पार, फिर तोरा करती प्रीत। नाव नदिया उस अड़ी, कभी न तोड़ें पतवार ॥",
    "पारस नदी का अजल, हेती सागर पार। पवन-धर सी सी हुई, बहती धार ॥",
    "कोई पता है मुझे, कुछ नहीं किसी का नाम। अपना ही है यहाँ, कहीं न जाने नाम ॥",
    "रहिमन एक पत्थर कहुँ, आपु बस लेहु बुझाय। जो हृदइ के पग ते भले, जाहिं उठि जाय ॥",
    "नंग-सोये कोटि दो, सबु बनाइ के साथ। इन श्मशान में चलत है, काहू कोटि देत ॥",
    "बाल ऊड़ि नीच परै, मन पै इतराय। चढ़त-परमन मारिये, समुझि चढ़्यौ जाय ॥",
    "दिल में एक ही नहीं, कोई कौन किसलय का ठौर। वह रहा है, कहाँ कहीं एक ॥",
    "सागर के जल में उठे, काले आकाश। जैसे पर्वत पार हैं, मोती सागर का छोर ॥",
    "दीपक सूरज का करे, दीपों की किरण। साँझ सुबह से हो रहे, तम-दिन अँधियारे रात ॥",
    "नदी पहाड़ चढ़ता नहीं, पर्वत नदिया धारा। पत्थर मीन सूखती, रेत गये सब रेत ॥",
    "जो जानै सब जगत सो, होय न जानि सकैं कोय। अपने कोहू मानिया, तेहि कुंभ लेय ॥",
    "माटी की धूल में, भूखी रख एक फूल। फूल सभी फूलों से मिले, सी होता आकाश ॥",
    "दो राह के पार पर, जाना कठिनगार। छोड़ नहीं कोई नहीं, अपना सरवार ॥",
    "संसार में मत भूलिये, जग के जगत बसाय। इसमें सेवक बने, सब जीवन होय ॥",
    "देखत न घन की नज़र, जो देखौ तेरी खोज। नहिं दीखे ओर के, कोई कोई दोष ॥",
    "प्रेम प्रेम भक्ति नित, हृदय बसत हरष। परमातम पिय नेह बिन, मिले न मिलता प्यार ॥",
    "जब तोहि जलाइये, तब तकइ आपहिं। तुम ही दृगन सों भरी, इन आँखों तें जाहि ॥",
    "धरती पर तन भूमि है, अब भू की देह। काँटों के पंखुने से नहीं, पंख पंखों की घास ॥",
    "मूल्य व्याकरण से हुआ, वहाँ सा मूल यह रूप। जहाँ कहीं से होता रहा, वह ही एक अंग ॥",
    "दिन भर की नींद से, थी दिन भर भी जाग। रवि-दिन तुम रख कर सके, सूर्य से अपने हाथ ॥",
    "नदी सागर-हर धार का, पार पार धार। जल जल में भर रही, मर गया पार ॥",
    "पत्थर बिन केवल नहीं, बिना बियोग संसार। धरनी तजै आपनाइया, ईश्वर का वास ॥",
    "पवन चले चलत जो सदा, जहाँ और जगह न कोय। गति हवा नदी को देख कर, पवन रही देय ॥",
    "धरती है बूँद-सा, जल है सागर समान। धरती पर जलधार है, धरती शीतल बरसात ॥",
    "धरती से देह सी, काया कातरित अंग। कबीर कुल्हाड़ी में पड़े, जल में पानी चंग ॥",
    "दोपहर हवा छाँव है, चाँदनी सु शीतल धूप। छाया दिन भर दे रही, दो पल शीतल दीप ॥",
    "दीपो दीपक जलता सदा, जले रात दिन रात। अँधियारे से सूर्य से, जगमगा उजियार ॥",
    "पोटली नयन की पटी, मन आइल सब मीत। बसत तेरे रंग है, होन मित्र सुजान ॥",
    "नदी पारै पारस की, गहि नाव तीर। पिय-मपधारि गई अधीर को, चली सैन ॥",
    "अब तक है एक लौं जल, आगत ही जल धार। कमल कीरी कमल में, धरि परै जल धार ॥",
    "कपट कथरा चलत बकरी, कर्म कोटि काठ। कहैं कबीर सो सफेद है, फिर न कोई होय ॥",
]

print(f"{'#':<4} {'m1':>4} {'m2':>4} {'m3':>4} {'m4':>4} {'Total':>6} {'MAS':>5} {'Valid':>6}")
print("=" * 60)

results = []
for i, doha in enumerate(doha_list, 1):
    # Each line is checked independently for comma
    cm = get_charan_matras(doha)
    mas = compute_mas(cm)
    cm4 = (cm + [0]*4)[:4]
    total = sum(cm4)
    results.append({'mas': mas, 'total': total})

    valid = "✓" if mas == 0 else "✗"
    print(f"{i:<4} {cm4[0]:>4} {cm4[1]:>4} {cm4[2]:>4} {cm4[3]:>4} {total:>6} {mas:>5} {valid:>6}")

print("=" * 60)
avg_mas = sum(r['mas'] for r in results) / len(results) if results else 0
print(f"Average MAS: {avg_mas:.2f}")

# --- Debug: Show word-by-word breakdown with line-wise comma detection ---
print("\n" + "="*70)
print("DEBUG: Line-by-line parsing (comma detection per line)")
print("="*70)
doha_test = doha_list[2]
text = doha_test.replace('॥', '।').strip()
lines = [l.strip() for l in text.split('।') if l.strip()]

for line_idx, line in enumerate(lines, 1):
    comma_present = has_comma_in_line(line)
    parse_mode = "WITH commas" if comma_present else "Word-by-word accumulation"
    print(f"\nLine {line_idx}: {line}")
    print(f"  Parse Mode: {parse_mode}")
    
    line_matras = parse_single_line(line, has_comma=None)
    print(f"  Charans: {line_matras}")
    
    if not comma_present:
        # Show detailed word breakdown for non-comma lines
        words = line.split()
        current_charan = 0
        charan_num = 1
        
        for word in words:
            word_matra = count_matra(word)
            current_charan += word_matra
            status = ""
            
            if current_charan >= 13:
                status = f" → CHARAN {charan_num} COMPLETE (total: {current_charan})"
                charan_num += 1
                current_charan = 0
            
            print(f"    '{word}' → {word_matra} matra | Running total: {current_charan}{status}")
        
        if current_charan > 0:
            print(f"    *** Remaining: {current_charan} matra ***")

#      m1   m2   m3   m4  Total   MAS  Valid
1      13   10   13   13     49     3      ✗
2      13   11   17    9     50     6      ✗
3      13   12   12   10     47     3      ✗
4      14   10   11    9     44     6      ✗
5      13   14   12    9     48     6      ✗
6      12   12   10   12     46     6      ✗
7      13    9   15    9     46     6      ✗
8      14   12   13   10     49     3      ✗
9      11   11   13    8     43     5      ✗
10     11   11   13   11     46     2      ✗
11     11   10   13   11     45     3      ✗
12     13   11   16   11     51     3      ✗
13     14   10   13   11     48     2      ✗
14     11    9   10   11     41     7      ✗
15     11    9   14   11     45     5      ✗
16     12   10   14    9     45     5      ✗
17     12    9   13   10     44     4      ✗
18     13   10   16   10     49     5      ✗
19     14   10   12    9     45     5      ✗
20     11   11   16   10     48     6      ✗
21     13    8   15    8     44     8      ✗
22     14 